# Phase 1 — repeated held-out target splits

The referee's demand: *train a model in separate data and test in a held-out dataset*. This notebook
does that inside ChEMBL, 200 times, and separates two questions that the single phrase "optimism"
runs together:

1. **Does the published definition hold up out of sample?** Freeze PAV + 2–5 therapeutic areas,
   estimate it on half the targets, evaluate on the other half. No selection anywhere.
2. **How much would a threshold search have inflated it?** Search for the OR-maximising window on
   half A, evaluate that window on half B, and measure the drop. This is the quantity the referee is
   actually asking about.

## Design decisions and why

**Split by target, never by target–indication pair.** PAV status, gPS and therapeutic-area count are
properties of the *gene*. A pair-level split would put the same gene's features on both sides, so the
"held-out" half would carry the very quantity that was selected on — near-total leakage, and the
first thing a referee would check. Every target appears in exactly one half.

**Repeated splits, not one.** With 51 approved supported pairs, a single 50/50 split leaves ~26 per
half and the answer is a coin flip: a lucky split proves nothing and an unlucky one disproves
nothing. 200 independent splits turn that into a distribution.

**The search space is a modelling choice, and it decides the answer.** Optimism is a property of the
procedure being simulated, not of the data. An unconstrained search over every window — including
single-value windows resting on five approved pairs — is a *worse* procedure than the one that
produced the paper, so it yields more optimism. Both are reported:

| Search space | What it models |
| ------------ | -------------- |
| `naive` | any window with ≥ 5 approved supported pairs in half A. The referee's literal charge, "picked the thresholds that maximise the result", with no judgement applied. An upper bound on optimism. |
| `reportable` | windows at least 3 units wide with ≥ 20 approved supported pairs in half A. What someone writing a paper would actually be willing to publish. |

Intermediate constraint levels are reported too, so the reader sees how optimism scales with the
freedom of the search rather than one number chosen by us.

**Labelling.** Re-observing the two published premises in half A is *stability*, not independence —
they were established on this same data. Only select-on-A / evaluate-on-B is a genuine held-out test.
The two are kept separate throughout.

## Pre-specified success criterion

Primary: the held-out OR interval excludes the all-GWAS baseline OR = 3.62. An interval excluding 1
would be nearly meaningless here — the question is whether the strict definition beats plain GWAS
support, not whether it beats nothing. Also reported against the mixed-effects-adjusted 3.14 and
against 1.
Secondary: 2–5 is selected in a majority of splits.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import chi2

from or10_stats import or_rs, support_mask, window_label

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

path_to_intermediate_data_folder = "../../../data/intermediate_files/"
master = pd.read_parquet(path_to_intermediate_data_folder + "ti_pairs_chembl_master-r1.parquet")

# statsmodels needs numeric, non-null predictors; targets absent from genes_therapeutic_areas have no
# disease GWAS association at all, which is a therapeutic-area count of zero
master["ta"] = master["uniqueTherapeuticAreas"].fillna(0).astype(float)
master["support_all"] = support_mask(master).astype(int)
master["support_pav"] = support_mask(master, pav=True).astype(int)

N_SPLITS = 200
SEED = 20260811
BASELINE_OR = 3.618578  # published all-GWAS support
ADJUSTED_BASELINE_OR = 3.14  # published mixed-effects estimate, adjusted for TA and sample size
PUBLISHED_OR = 10.288962
PUBLISHED_OR_CI = (6.707865, 15.781881)
PUBLISHED_RS = 4.843708
PUBLISHED_RS_CI = (4.051253, 5.791173)
PUBLISHED_WINDOW = (2, 5)
PUB_LABEL = window_label(*PUBLISHED_WINDOW)

print(master.shape)
print("targets:", master["targetId"].nunique())
print("approved pairs:", int(master["approved"].sum()))
print(
    "approved pairs with PAV + 2-5 TA support:",
    int(master.loc[support_mask(master, pav=True, ta_min=2, ta_max=5), "approved"].sum()),
)

(37377, 13)
targets: 1273
approved pairs: 4564
approved pairs with PAV + 2-5 TA support: 51


## Candidate windows and the fast odds ratio

`scipy.stats.fisher_exact` returns the sample odds ratio `ad/bc`, which is what the published
enrichment reports, so the search can use plain arithmetic and skip the exact test — verified against
`or_rs` below. Every reported number still goes through `or_rs`.

In [2]:
CANDIDATE_WINDOWS = [(lo, hi) for lo in range(1, 7) for hi in list(range(lo, 11)) + [None]]
assert PUBLISHED_WINDOW in CANDIDATE_WINDOWS
print("candidate windows:", len(CANDIDATE_WINDOWS))


def window_width(lo, hi):
    """Number of therapeutic-area values a window admits; unbounded windows count as wide."""
    return 99 if hi is None else hi - lo + 1


def fast_or(support, approved):
    """Sample odds ratio ad/bc, plus the cell counts."""
    s = support.to_numpy(dtype=bool) if isinstance(support, pd.Series) else np.asarray(support, dtype=bool)
    y = approved.to_numpy() if isinstance(approved, pd.Series) else np.asarray(approved)
    x_g, n_g = int(y[s].sum()), int(s.sum())
    x_n, n_n = int(y[~s].sum()), int((~s).sum())
    a, b, c, d = n_n - x_n, x_n, n_g - x_g, x_g
    odds = np.nan if (b == 0 or c == 0) else (a * d) / (b * c)
    return odds, x_g, n_g


check_mask = support_mask(master, pav=True, ta_min=2, ta_max=5)
odds, x_g, n_g = fast_or(check_mask, master["approved"])
reference = or_rs(check_mask, master["approved"])
assert np.isclose(odds, reference["odds_ratio"]), (odds, reference["odds_ratio"])
assert x_g == reference["yes_evid-high_clinphase"]
print(f"fast_or agrees with or_rs: OR = {odds:.6f}, {x_g} approved of {n_g} supported pairs")

candidate windows: 51
fast_or agrees with or_rs: OR = 10.288962, 51 approved of 87 supported pairs


## The 200 splits

Targets are stratified by how many approved pairs they carry, then alternately assigned within each
stratum. That balances the scarce quantity — approved pairs — between halves without ever splitting a
target. The same 200 partitions are reused by every search space, so differences between search
spaces are not split noise.

In [3]:
approved_per_target = master.groupby("targetId")["approved"].sum()


def target_split(rng):
    """Assign every target to half A or B, stratified on approved-pair count."""
    half_a, half_b = [], []
    for _, targets in approved_per_target.groupby(approved_per_target.values):
        ids = targets.index.to_numpy()
        rng.shuffle(ids)
        start = rng.integers(2)  # random starting side so no stratum is systematically biased
        for i, target in enumerate(ids):
            (half_a if (i + start) % 2 == 0 else half_b).append(target)
    return set(half_a), set(half_b)


rng = np.random.default_rng(SEED)
SPLITS = []
for _ in range(N_SPLITS):
    a_ids, b_ids = target_split(rng)
    assert not (a_ids & b_ids)
    idx_a = master["targetId"].isin(a_ids).to_numpy()
    SPLITS.append((master[idx_a], master[~idx_a]))

balance = pd.DataFrame(
    [
        {
            "split": i,
            "targets_a": df_a["targetId"].nunique(),
            "pairs_a": len(df_a),
            "approved_a": int(df_a["approved"].sum()),
            "approved_b": int(df_b["approved"].sum()),
            "approved_strict_a": int(df_a.loc[support_mask(df_a, pav=True, ta_min=2, ta_max=5), "approved"].sum()),
            "approved_strict_b": int(df_b.loc[support_mask(df_b, pav=True, ta_min=2, ta_max=5), "approved"].sum()),
        }
        for i, (df_a, df_b) in enumerate(SPLITS)
    ]
)
print(
    balance[["targets_a", "pairs_a", "approved_a", "approved_b", "approved_strict_a", "approved_strict_b"]]
    .describe()
    .round(1)
    .to_string()
)

       targets_a  pairs_a  approved_a  approved_b  approved_strict_a  approved_strict_b
count      200.0    200.0       200.0       200.0              200.0              200.0
mean       636.5  18659.6      2278.0      2286.0               25.4               25.6
std          2.5    605.5        96.7        96.7                5.5                5.5
min        630.0  16644.0      2046.0      1978.0               11.0               14.0
25%        635.0  18268.0      2207.0      2219.8               22.0               21.0
50%        636.0  18744.0      2277.5      2286.5               26.0               25.0
75%        638.0  19091.8      2344.2      2357.0               30.0               29.0
max        644.0  20451.0      2586.0      2518.0               37.0               40.0


## Question 1 — the published definition, held out

No selection: the frozen 2–5 window is estimated on half A and on half B of every split. If the
window were a fluke of the full data, halving the targets would move it; and half B is untouched by
any choice, so its distribution is the honest out-of-sample performance of the published definition.

In [4]:
fixed_rows = []
for i, (df_a, df_b) in enumerate(SPLITS):
    row = {"split": i}
    for half_name, df_half in (("a", df_a), ("b", df_b)):
        res = or_rs(
            support_mask(df_half, pav=True, ta_min=PUBLISHED_WINDOW[0], ta_max=PUBLISHED_WINDOW[1]),
            df_half["approved"],
        )
        degenerate = res["yes_evid-high_clinphase"] == 0 or res["yes_evid-low_clinphase"] == 0
        row[f"or_{half_name}"] = np.nan if degenerate else res["odds_ratio"]
        row[f"rs_{half_name}"] = np.nan if degenerate else res["relative_success"]
        row[f"ci_low_{half_name}"] = res["ci_low"]
        row[f"n_approved_support_{half_name}"] = res["yes_evid-high_clinphase"]
        row[f"degenerate_{half_name}"] = degenerate
    baseline_b = or_rs(support_mask(df_b), df_b["approved"])
    row["or_b_all_gwas"] = baseline_b["odds_ratio"]
    fixed_rows.append(row)

fixed = pd.DataFrame(fixed_rows)
print(
    f"splits where the frozen window gives an inestimable OR (empty 2x2 cell): "
    f"A {int(fixed['degenerate_a'].sum())}, B {int(fixed['degenerate_b'].sum())}"
)
print(fixed[["or_a", "or_b", "rs_b", "n_approved_support_b", "or_b_all_gwas"]].describe().round(3).to_string())

splits where the frozen window gives an inestimable OR (empty 2x2 cell): A 0, B 0
          or_a     or_b     rs_b  n_approved_support_b  or_b_all_gwas
count  200.000  200.000  200.000               200.000        200.000
mean    10.420   10.696    4.861                25.560          3.654
std      2.539    2.483    0.472                 5.529          0.356
min      4.977    5.051    3.383                14.000          2.828
25%      8.779    9.012    4.561                21.000          3.374
50%     10.239   10.324    4.860                25.000          3.644
75%     11.822   12.161    5.156                29.000          3.890
max     20.481   17.928    5.993                40.000          4.681


## Question 2 — how much a threshold search inflates the estimate

`run_search` simulates one search space across all 200 splits: find the OR-maximising window on half
A subject to the constraints, then evaluate that frozen window on half B. `min_approved` is the
minimum number of approved supported pairs a window must have in half A to be selectable, and
`min_width` the minimum number of therapeutic-area values it must span.

In [5]:
def run_search(min_approved, min_width, label):
    """Select the OR-maximising window on half A, evaluate it on half B, for every split."""
    rows = []
    for i, (df_a, df_b) in enumerate(SPLITS):
        best = None
        for lo, hi in CANDIDATE_WINDOWS:
            if window_width(lo, hi) < min_width:
                continue
            odds, x_g, n_g = fast_or(support_mask(df_a, pav=True, ta_min=lo, ta_max=hi), df_a["approved"])
            if np.isnan(odds) or x_g < min_approved:
                continue
            if best is None or odds > best[0]:
                best = (odds, lo, hi, x_g, n_g)
        if best is None:
            rows.append({"search_space": label, "split": i, "selected_window": None, "selectable": False})
            continue

        or_a, lo, hi, x_g_a, n_g_a = best
        rs_a = or_rs(support_mask(df_a, pav=True, ta_min=lo, ta_max=hi), df_a["approved"])["relative_success"]
        held = or_rs(support_mask(df_b, pav=True, ta_min=lo, ta_max=hi), df_b["approved"])
        # or_rs returns OR = 1 when a cell is empty; that is a missing estimate, not a null result
        degenerate = held["yes_evid-high_clinphase"] == 0 or held["yes_evid-low_clinphase"] == 0
        rows.append(
            {
                "search_space": label,
                "split": i,
                "selectable": True,
                "selected_window": window_label(lo, hi),
                "selected_ta_min": lo,
                "selected_ta_max": hi,
                "or_a": or_a,
                "rs_a": rs_a,
                "n_approved_support_a": x_g_a,
                "n_support_a": n_g_a,
                "or_b": np.nan if degenerate else held["odds_ratio"],
                "rs_b": np.nan if degenerate else held["relative_success"],
                "or_b_ci_low": held["ci_low"],
                "n_approved_support_b": held["yes_evid-high_clinphase"],
                "degenerate_b": degenerate,
            }
        )
    return pd.DataFrame(rows)


SEARCH_SPACES = [
    (5, 1, "naive (>=5 approved, any width)"),
    (10, 1, "10 approved, any width"),
    (20, 1, "20 approved, any width"),
    (20, 3, "reportable (>=20 approved, width >=3)"),
    (30, 3, "30 approved, width >=3"),
]
searches = pd.concat([run_search(m, w, label) for m, w, label in SEARCH_SPACES], ignore_index=True)

coverage = searches.groupby("search_space", sort=False).agg(
    splits=("split", "size"),
    no_window_selectable=("selectable", lambda s: int((~s).sum())),
    held_out_inestimable=("degenerate_b", lambda s: int(s.eq(True).sum())),
    median_approved_support_b=("n_approved_support_b", "median"),
)
print(coverage.to_string())

                                       splits  no_window_selectable  held_out_inestimable  median_approved_support_b
search_space                                                                                                        
naive (>=5 approved, any width)           200                     0                     9                        5.0
10 approved, any width                    200                     0                     5                        8.0
20 approved, any width                    200                     2                     0                       21.0
reportable (>=20 approved, width >=3)     200                     2                     0                       21.0
30 approved, width >=3                    200                    31                     0                       28.0


### Which window the search picks

The published window's selection frequency, per search space. Under an unconstrained search the
winner is almost always a narrow window sitting on a handful of approved pairs — which is exactly why
that search space overstates the optimism of the published procedure.

In [6]:
for label in searches["search_space"].unique():
    sub = searches[(searches["search_space"] == label) & searches["selectable"]]
    freq = sub["selected_window"].value_counts()
    top = ", ".join(f"{w} {100 * n / len(sub):.0f}%" for w, n in freq.head(6).items())
    print(f"{label}")
    print(
        f"  {PUB_LABEL} selected in {100 * (sub['selected_window'] == PUB_LABEL).mean():.1f}% of splits"
        f" | median approved pairs behind the selected window in A: {sub['n_approved_support_a'].median():.0f}"
    )
    print(f"  most frequent: {top}")

window_frequency = (
    searches[searches["selectable"]]
    .groupby(["search_space", "selected_window"], sort=False)
    .size()
    .rename("n")
    .reset_index()
)
window_frequency["pct"] = window_frequency.groupby("search_space")["n"].transform(lambda s: 100 * s / s.sum())

naive (>=5 approved, any width)
  2-5 selected in 0.0% of splits | median approved pairs behind the selected window in A: 7
  most frequent: 2 35%, 5 18%, 4 14%, 2-3 14%, 3 8%, 1-2 4%
10 approved, any width
  2-5 selected in 7.5% of splits | median approved pairs behind the selected window in A: 13
  most frequent: 2-3 28%, 4 20%, 4-5 14%, 2-4 10%, 2-5 8%, 3-4 4%
20 approved, any width
  2-5 selected in 31.3% of splits | median approved pairs behind the selected window in A: 23
  most frequent: 2-5 31%, 2-4 25%, 4-5 11%, 2-7 6%, 2-6 5%, 3-4 4%
reportable (>=20 approved, width >=3)
  2-5 selected in 34.8% of splits | median approved pairs behind the selected window in A: 24
  most frequent: 2-5 35%, 2-4 31%, 4-6 6%, 2-7 6%, 4-7 5%, 2-6 5%
30 approved, width >=3
  2-5 selected in 24.9% of splits | median approved pairs behind the selected window in A: 31
  most frequent: 2-5 25%, 2-7 15%, 2-6 15%, >=2 14%, 2-8 6%, >=1 5%


### Held-out distributions and optimism

`optimism_factor` is `exp(mean(log OR_A − log OR_B))`, the multiplicative scale appropriate to a
ratio; `optimism_additive` is the plain difference of means for readers who want it. The interval on
the factor is the **Monte Carlo standard error of the mean across splits** — it quantifies how well
200 splits pin down the expected optimism, and is *not* a confidence interval accounting for sampling
of the underlying data. The split-to-split spread is reported separately, and is much wider.

In [7]:
def summarise(values):
    """Median and 2.5/97.5 percentiles of a distribution across splits, ignoring missing values."""
    v = pd.Series(values).replace([np.inf, -np.inf], np.nan).dropna()
    return v.mean(), v.median(), np.percentile(v, 2.5), np.percentile(v, 97.5), len(v)


def optimism_row(df, label):
    """In-sample versus held-out summary for one search space (or the frozen window)."""
    pair = df[["or_a", "or_b"]].replace([np.inf, -np.inf], np.nan).dropna()
    d = np.log(pair["or_a"]) - np.log(pair["or_b"])
    factor = float(np.exp(d.mean()))
    se = float(d.std(ddof=1) / np.sqrt(len(d)))
    mean_b, median_b, lo_b, hi_b, n_b = summarise(df["or_b"])
    rs_pair = df[["rs_a", "rs_b"]].replace([np.inf, -np.inf], np.nan).dropna()
    rs_d = np.log(rs_pair["rs_a"]) - np.log(rs_pair["rs_b"])
    return {
        "procedure": label,
        "n_splits_used": len(pair),
        "mean_or_in_sample": float(pair["or_a"].mean()),
        "mean_or_held_out": float(pair["or_b"].mean()),
        "median_or_held_out": median_b,
        "held_out_spread_2.5": lo_b,
        "held_out_spread_97.5": hi_b,
        "optimism_additive": float((pair["or_a"] - pair["or_b"]).mean()),
        "optimism_factor": factor,
        "optimism_factor_mcse_low": float(np.exp(d.mean() - 1.96 * se)),
        "optimism_factor_mcse_high": float(np.exp(d.mean() + 1.96 * se)),
        "optimism_factor_spread_2.5": float(np.exp(np.percentile(d, 2.5))),
        "optimism_factor_spread_97.5": float(np.exp(np.percentile(d, 97.5))),
        "rs_optimism_factor": float(np.exp(rs_d.mean())) if len(rs_d) else np.nan,
        "corrected_or": PUBLISHED_OR / factor,
        "corrected_or_from_published_ci_low": PUBLISHED_OR_CI[0] / factor,
        "corrected_or_from_published_ci_high": PUBLISHED_OR_CI[1] / factor,
        "corrected_rs": PUBLISHED_RS / float(np.exp(rs_d.mean())) if len(rs_d) else np.nan,
    }


optimism = pd.DataFrame(
    [optimism_row(fixed, "frozen published 2-5 window (no selection)")]
    + [
        optimism_row(searches[(searches["search_space"] == label) & searches["selectable"]], label)
        for label in searches["search_space"].unique()
    ]
)
optimism[
    [
        "procedure",
        "n_splits_used",
        "mean_or_in_sample",
        "mean_or_held_out",
        "median_or_held_out",
        "held_out_spread_2.5",
        "held_out_spread_97.5",
    ]
].round(3)

,procedure,n_splits_used,mean_or_in_sample,mean_or_held_out,median_or_held_out,held_out_spread_2.5,held_out_spread_97.5
0,frozen published 2-5 window (no selection),200,10.420,10.696,10.324,6.618,16.752
1,"naive (>=5 approved, any width)",191,22.284,9.172,7.274,1.467,28.019
2,"10 approved, any width",195,13.932,7.893,7.428,2.802,15.127
3,"20 approved, any width",198,11.002,8.877,8.635,4.490,13.948
4,"reportable (>=20 approved, width >=3)",198,10.857,9.037,8.743,4.958,14.073
5,"30 approved, width >=3",169,9.152,8.154,7.966,4.899,13.824


In [8]:
optimism[
    [
        "procedure",
        "optimism_additive",
        "optimism_factor",
        "optimism_factor_mcse_low",
        "optimism_factor_mcse_high",
        "optimism_factor_spread_2.5",
        "optimism_factor_spread_97.5",
        "corrected_or",
        "corrected_or_from_published_ci_low",
        "corrected_or_from_published_ci_high",
        "corrected_rs",
    ]
].round(3)

,procedure,optimism_additive,optimism_factor,optimism_factor_mcse_low,optimism_factor_mcse_high,optimism_factor_spread_2.5,optimism_factor_spread_97.5,corrected_or,corrected_or_from_published_ci_low,corrected_or_from_published_ci_high,corrected_rs
0,frozen published 2-5 window (no selection),-0.276,0.972,0.910,1.037,0.371,2.572,10.590,6.904,16.243,4.905
1,"naive (>=5 approved, any width)",13.112,2.633,2.316,2.993,0.498,14.143,3.908,2.548,5.995,3.259
2,"10 approved, any width",6.040,1.783,1.622,1.961,0.450,7.808,5.771,3.762,8.851,3.757
3,"20 approved, any width",2.125,1.235,1.147,1.330,0.463,3.295,8.331,5.431,12.778,4.421
4,"reportable (>=20 approved, width >=3)",1.820,1.192,1.113,1.277,0.463,3.169,8.631,5.627,13.238,4.496
5,"30 approved, width >=3",0.999,1.105,1.034,1.180,0.505,2.632,9.315,6.073,14.288,4.650


### Success criterion

Evaluated against three bars, for the frozen published window and for each search space. `interval`
is the 2.5–97.5 percentile range of the held-out odds ratio across splits; `splits_with_ci_above_bar`
counts the splits whose own 95% CI lower bound clears the bar.

In [9]:
bars = {"all-GWAS baseline 3.62": BASELINE_OR, "adjusted baseline 3.14": ADJUSTED_BASELINE_OR, "no effect 1.0": 1.0}
sources = [("frozen published 2-5 window", fixed, "or_b", "ci_low_b")] + [
    (label, searches[(searches["search_space"] == label) & searches["selectable"]], "or_b", "or_b_ci_low")
    for label in searches["search_space"].unique()
]

criterion_rows = []
for bar_name, bar in bars.items():
    for label, df, or_col, ci_col in sources:
        v = df[or_col].replace([np.inf, -np.inf], np.nan).dropna()
        criterion_rows.append(
            {
                "bar": bar_name,
                "procedure": label,
                "median_held_out_or": v.median(),
                "interval_2.5": np.percentile(v, 2.5),
                "interval_excludes_bar": bool(np.percentile(v, 2.5) > bar),
                "splits_above_bar_pct": 100 * (v > bar).mean(),
                "splits_with_ci_above_bar_pct": 100 * (df[ci_col].dropna() > bar).mean(),
            }
        )
criteria = pd.DataFrame(criterion_rows)
criteria.round(3)

,bar,procedure,median_held_out_or,interval_2.5,interval_excludes_bar,splits_above_bar_pct,splits_with_ci_above_bar_pct
0,all-GWAS baseline 3.62,frozen published 2-5 window,10.324,6.618,True,100.000,95.500
1,all-GWAS baseline 3.62,"naive (>=5 approved, any width)",7.274,1.467,False,89.529,8.901
2,all-GWAS baseline 3.62,"10 approved, any width",7.428,2.802,False,94.872,35.897
3,all-GWAS baseline 3.62,"20 approved, any width",8.635,4.490,True,99.495,73.737
4,all-GWAS baseline 3.62,"reportable (>=20 approved, width >=3)",8.743,4.958,True,100.000,79.798
5,all-GWAS baseline 3.62,"30 approved, width >=3",7.966,4.899,True,100.000,86.982
6,adjusted baseline 3.14,frozen published 2-5 window,10.324,6.618,True,100.000,98.500
7,adjusted baseline 3.14,"naive (>=5 approved, any width)",7.274,1.467,False,91.623,23.037
8,adjusted baseline 3.14,"10 approved, any width",7.428,2.802,False,96.410,46.667
9,adjusted baseline 3.14,"20 approved, any width",8.635,4.490,True,100.000,83.333


## Premise stability across half-samples

Not a held-out test. Premises 1 and 2 were established on this data, so re-observing them in a random
half shows only that they do not depend on a particular subset.

**Premise 1 — PAV enrichment.** Logistic regression with a three-level evidence factor (no support,
support without a PAV, support with a PAV), coefficients compared by a t-test on the contrast; the
published values are OR 6.0 versus 3.1, P = 2e-4 (`02-enrichment-groups.ipynb`).

**Premise 2 — non-linear pleiotropy.** `log(TA + 1)` and its square added to
`outcome ~ geneticSupport`, nested models compared by likelihood ratio; published LR = 64.9,
P = 7.9e-16 (`11-non-linearity-gPS.ipynb`).

In [10]:
def premise_pav(df):
    """Premise 1 on one half: PAV support versus support without a PAV."""
    e = np.where(df["support_all"] == 0, 0, np.where(df["support_pav"] == 1, 2, 1))
    data = pd.DataFrame({"outcome": df["approved"].to_numpy(), "E": e})
    if len(np.unique(e)) < 3 or data.groupby("E")["outcome"].sum().min() == 0:
        return {"or_nonpav": np.nan, "or_pav": np.nan, "p_diff": np.nan, "pav_higher": False}
    fit = smf.logit("outcome ~ C(E)", data=data).fit(disp=False)
    contrast = np.zeros(len(fit.params))
    contrast[1], contrast[2] = 1, -1
    return {
        "or_nonpav": float(np.exp(fit.params.iloc[1])),
        "or_pav": float(np.exp(fit.params.iloc[2])),
        "p_diff": float(np.ravel(fit.t_test(contrast).pvalue)[0]),
        "pav_higher": float(fit.params.iloc[2]) > float(fit.params.iloc[1]),
    }


def premise_nonlinearity(df):
    """Premise 2 on one half: likelihood-ratio test for a quadratic log(TA + 1) term."""
    data = pd.DataFrame(
        {
            "outcome": df["approved"].to_numpy(),
            "geneticSupport": df["support_all"].to_numpy(),
            "logta": np.log(df["ta"].to_numpy() + 1),
        }
    )
    data["logta2"] = data["logta"] ** 2
    m0 = smf.logit("outcome ~ geneticSupport", data=data).fit(disp=False)
    m1 = smf.logit("outcome ~ geneticSupport + logta", data=data).fit(disp=False)
    m2 = smf.logit("outcome ~ geneticSupport + logta + logta2", data=data).fit(disp=False)
    lr_21 = 2 * (float(m2.llf) - float(m1.llf))
    b, a = float(m2.params["logta"]), float(m2.params["logta2"])
    return {
        "lr_quadratic": lr_21,
        "p_quadratic": float(chi2.sf(lr_21, 1)),
        "p_both_terms": float(chi2.sf(2 * (float(m2.llf) - float(m0.llf)), 2)),
        "peak_ta": float(np.exp(-b / (2 * a)) - 1) if a != 0 else np.nan,
    }


full_p1, full_p2 = premise_pav(master), premise_nonlinearity(master)
print(
    f"premise 1 (full data): OR PAV = {full_p1['or_pav']:.3f}, OR non-PAV = {full_p1['or_nonpav']:.3f}, "
    f"difference p = {full_p1['p_diff']:.3g}"
)
print(
    f"premise 2 (full data): quadratic LR = {full_p2['lr_quadratic']:.3f}, p = {full_p2['p_quadratic']:.3g}, "
    f"fitted peak at {full_p2['peak_ta']:.2f} therapeutic areas"
)
assert np.isclose(full_p2["lr_quadratic"], 64.897, atol=0.05), full_p2["lr_quadratic"]
assert full_p1["or_pav"] > full_p1["or_nonpav"]

premise 1 (full data): OR PAV = 6.048, OR non-PAV = 3.092, difference p = 0.000244
premise 2 (full data): quadratic LR = 64.897, p = 7.89e-16, fitted peak at 1.90 therapeutic areas


In [11]:
premise_rows = []
for i, (df_a, _) in enumerate(SPLITS):
    row = {"split": i}
    row.update({f"p1_{k}": v for k, v in premise_pav(df_a).items()})
    row.update({f"p2_{k}": v for k, v in premise_nonlinearity(df_a).items()})
    premise_rows.append(row)
premises = pd.DataFrame(premise_rows)

stability = pd.DataFrame(
    [
        {"premise": "1: PAV OR > non-PAV OR (direction)", "holds_pct": 100 * premises["p1_pav_higher"].mean()},
        {"premise": "1: difference p < 0.05", "holds_pct": 100 * (premises["p1_p_diff"] < 0.05).mean()},
        {"premise": "2: quadratic LRT p < 0.05", "holds_pct": 100 * (premises["p2_p_quadratic"] < 0.05).mean()},
        {"premise": "2: quadratic LRT p < 1e-4", "holds_pct": 100 * (premises["p2_p_quadratic"] < 1e-4).mean()},
        {
            "premise": "both premises at p < 0.05",
            "holds_pct": 100 * ((premises["p1_p_diff"] < 0.05) & (premises["p2_p_quadratic"] < 0.05)).mean(),
        },
    ]
)
print(stability.round(1).to_string(index=False))
print()
print("half-sample premise estimates:")
print(
    premises[["p1_or_pav", "p1_or_nonpav", "p1_p_diff", "p2_lr_quadratic", "p2_peak_ta"]]
    .describe()
    .round(3)
    .to_string()
)

                           premise  holds_pct
1: PAV OR > non-PAV OR (direction)       99.5
            1: difference p < 0.05       79.5
         2: quadratic LRT p < 0.05      100.0
         2: quadratic LRT p < 1e-4       92.5
         both premises at p < 0.05       79.5

half-sample premise estimates:
       p1_or_pav  p1_or_nonpav  p1_p_diff  p2_lr_quadratic  p2_peak_ta
count    200.000       200.000    200.000          200.000     200.000
mean       6.096         3.098      0.049           34.125       1.912
std        1.110         0.349      0.120           15.623       0.326
min        3.125         2.119      0.000            4.415       1.039
25%        5.375         2.860      0.002           22.562       1.710
50%        6.064         3.077      0.011           33.340       1.873
75%        6.629         3.335      0.041           43.976       2.115
max       11.263         4.129      0.872           92.592       3.372


## Headline numbers

In [12]:
frozen = optimism.iloc[0]
naive = optimism[optimism["procedure"].str.startswith("naive")].iloc[0]
reportable = optimism[optimism["procedure"].str.startswith("reportable")].iloc[0]

print(
    f"published:                        OR {PUBLISHED_OR:.2f} [{PUBLISHED_OR_CI[0]:.2f}, {PUBLISHED_OR_CI[1]:.2f}], "
    f"RS {PUBLISHED_RS:.2f}"
)
print(f"all-GWAS baseline:                OR {BASELINE_OR:.2f}")
print()
print("frozen 2-5 window, no selection")
print(
    f"  held-out OR median {frozen['median_or_held_out']:.2f} "
    f"(split spread {frozen['held_out_spread_2.5']:.2f}-{frozen['held_out_spread_97.5']:.2f}), "
    f"optimism factor {frozen['optimism_factor']:.2f}"
)
print()
print("window chosen by search on half A, evaluated on half B")
print(
    f"  naive search:      in-sample {naive['mean_or_in_sample']:.2f} -> held out {naive['mean_or_held_out']:.2f}, "
    f"factor {naive['optimism_factor']:.2f} "
    f"[MC {naive['optimism_factor_mcse_low']:.2f}, {naive['optimism_factor_mcse_high']:.2f}], "
    f"corrected OR {naive['corrected_or']:.2f}"
)
print(
    f"  reportable search: in-sample {reportable['mean_or_in_sample']:.2f} -> held out "
    f"{reportable['mean_or_held_out']:.2f}, factor {reportable['optimism_factor']:.2f} "
    f"[MC {reportable['optimism_factor_mcse_low']:.2f}, {reportable['optimism_factor_mcse_high']:.2f}], "
    f"corrected OR {reportable['corrected_or']:.2f} "
    f"(from published CI: {reportable['corrected_or_from_published_ci_low']:.2f}-"
    f"{reportable['corrected_or_from_published_ci_high']:.2f}), corrected RS {reportable['corrected_rs']:.2f}"
)

published:                        OR 10.29 [6.71, 15.78], RS 4.84
all-GWAS baseline:                OR 3.62

frozen 2-5 window, no selection
  held-out OR median 10.32 (split spread 6.62-16.75), optimism factor 0.97

window chosen by search on half A, evaluated on half B
  naive search:      in-sample 22.28 -> held out 9.17, factor 2.63 [MC 2.32, 2.99], corrected OR 3.91
  reportable search: in-sample 10.86 -> held out 9.04, factor 1.19 [MC 1.11, 1.28], corrected OR 8.63 (from published CI: 5.63-13.24), corrected RS 4.50


## Export

In [13]:
fixed.to_csv(path_to_intermediate_data_folder + "or10_phase1_frozen_window_splits-r1.csv", index=False)
searches.to_csv(path_to_intermediate_data_folder + "or10_phase1_search_splits-r1.csv", index=False)
premises.to_csv(path_to_intermediate_data_folder + "or10_phase1_premises-r1.csv", index=False)
optimism.to_csv(path_to_intermediate_data_folder + "or10_phase1_optimism-r1.csv", index=False)
criteria.to_csv(path_to_intermediate_data_folder + "or10_phase1_criteria-r1.csv", index=False)
window_frequency.to_csv(path_to_intermediate_data_folder + "or10_phase1_window_frequency-r1.csv", index=False)
stability.to_csv(path_to_intermediate_data_folder + "or10_phase1_premise_stability-r1.csv", index=False)
balance.to_csv(path_to_intermediate_data_folder + "or10_phase1_split_balance-r1.csv", index=False)
print("exported 8 tables")

exported 8 tables
